# Incremental learning with SGD
When data does not comfortably fit into one training pass, the model must improve from batches. I track quality after each `partial_fit` instead of treating online learning as a black box.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score

X,y=make_classification(n_samples=6000,n_features=40,n_informative=12,random_state=42)
X_train,X_test=X[:5000],X[5000:]; y_train,y_test=y[:5000],y[5000:]
scaler=StandardScaler().fit(X_train)
X_train=scaler.transform(X_train); X_test=scaler.transform(X_test)
model=SGDClassifier(loss='log_loss',alpha=1e-4,random_state=42)
rows=[]
for epoch in range(4):
    for start in range(0,len(X_train),250):
        xb=X_train[start:start+250]; yb=y_train[start:start+250]
        model.partial_fit(xb,yb,classes=np.array([0,1]))
    p=model.predict_proba(X_test)[:,1]
    rows.append({'epoch':epoch+1,'log_loss':log_loss(y_test,p),'roc_auc':roc_auc_score(y_test,p)})
pd.DataFrame(rows).round(4)


## Operational thought
Incremental training adds a new question: is the data distribution stationary? A model that can update continuously can also continuously chase drift or feedback loops. Monitoring becomes part of the learning system.